In [1]:
import torch
from torch.utils.data import Dataset

PAD_ID = 2

def flatten_state(cur_seq):
    flat = []
    for seq in cur_seq:
        for b in seq:
            flat.append(1 if b > 0 else 0)
    return torch.tensor(flat, dtype=torch.long)

class PrefixNextBitPaddedDataset(Dataset):
    def __init__(self, samples):
        self.data = []
        for seq, label in samples:
            x = flatten_state(seq)
            y = 1 if label > 0 else 0
            self.data.append((x, y))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

In [2]:
def collate_pad(batch):
    """
    batch: list of (x, y) where x is [T_i] of 0/1 tokens
    returns:
      X: [B, T] int64 in {0,1,PAD_ID}
      Y: [B]     int64 in {0,1}
      M: [B, T]  bool   True where PAD
    """
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    padded, masks = [], []
    for x in xs:
        pad_len = max_len - x.size(0)
        if pad_len > 0:
            x_pad = torch.cat([x, torch.full((pad_len,), PAD_ID, dtype=torch.long)])
            m_pad = torch.cat([torch.zeros(x.size(0), dtype=torch.bool),
                               torch.ones(pad_len, dtype=torch.bool)])   # True = PAD
        else:
            x_pad = x
            m_pad = torch.zeros_like(x, dtype=torch.bool)
        padded.append(x_pad); masks.append(m_pad)

    X = torch.stack(padded)                   # [B, T]
    Y = torch.tensor(ys, dtype=torch.long)    # [B]
    M = torch.stack(masks)                    # [B, T] (True=PAD)
    return X, Y, M

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class PolicyOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=3,         # tokens {0,1,PAD_ID}
        pad_id=PAD_ID,
        max_len=4096,
        d_model=256,
        nhead=8,
        nlayers=4,
        dim_ff=512,
        dropout=0.1,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_len, d_model)  # learned positional embeddings

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, 2)  # logits for {0,1}

    def forward(self, x, pad_mask):
        """
        x:        [B, T] int64 in {0,1,pad_id}
        pad_mask: [B, T] bool, True where PAD (ignored by attention)
        """
        B, T = x.size()
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)  # [B, T]
        h = self.token_emb(x) + self.pos_emb(pos)                         # [B, T, D]

        # Mask pads inside the encoder
        h = self.encoder(h, src_key_padding_mask=pad_mask)                # [B, T, D]

        # Pool from the last non-pad token per sequence
        valid = ~pad_mask                                # True where real token
        lengths = valid.long().sum(dim=1) - 1            # index of last real token
        lengths = lengths.clamp(min=0)                   # guard against all-pad (shouldn't happen)
        idx = lengths.view(B, 1, 1).expand(B, 1, h.size(-1))
        last_h = h.gather(1, idx).squeeze(1)             # [B, D]

        logits = self.head(self.norm(last_h))            # [B, 2]
        return logits

In [4]:
from torch.utils.data import DataLoader
import torch.optim as optim

def train_overfit(samples, epochs=50, batch_size=32, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ds = PrefixNextBitPaddedDataset(samples)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=collate_pad)

    model = PolicyOnlyTransformer(vocab_size=3, pad_id=PAD_ID, max_len=4096).to(device)
    opt = optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)

    for ep in range(1, epochs+1):
        model.train()
        tot_loss = tot_acc = n = 0
        for X, Y, M in loader:
            X, Y, M = X.to(device), Y.to(device), M.to(device)
            logits = model(X, pad_mask=M)
            loss = F.cross_entropy(logits, Y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            with torch.no_grad():
                acc = (logits.argmax(-1) == Y).float().sum().item()
            bs = X.size(0)
            tot_loss += loss.item() * bs
            tot_acc  += acc
            n += bs
        if ep % 2 == 0:
            print(f"epoch {ep:3d} | loss {tot_loss/n:.4f} | acc {tot_acc/n:.3f}")

    return model

In [5]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

@torch.no_grad()
def evaluate_policy(model, dataset, batch_size=256, device=None, return_mistakes=False):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_pad)

    tot_loss = tot_correct = n = 0
    mistakes = []
    for X, Y, M in loader:
        X, Y, M = X.to(device), Y.to(device), M.to(device)
        logits = model(X, pad_mask=M)            # [B,2]
        loss = F.cross_entropy(logits, Y, reduction="sum")
        pred = logits.argmax(dim=-1)             # [B]
        print(pred)

        tot_loss    += loss.item()
        tot_correct += (pred == Y).sum().item()
        n           += X.size(0)

        if return_mistakes:
            wrong = (pred != Y).nonzero(as_tuple=False).squeeze(1)
            for i in wrong.tolist():
                mistakes.append({
                    "idx": n - X.size(0) + i,
                    "true": int(Y[i].item()),
                    "pred": int(pred[i].item()),
                    "prob": float(torch.softmax(logits[i], -1)[pred[i]].item()),
                })

    return {
        "loss": tot_loss / max(n, 1),
        "acc":  tot_correct / max(n, 1),
        "n": n,
        "mistakes": mistakes if return_mistakes else None,
    }

In [6]:
import matplotlib.pyplot as plt
from torch.utils.data import random_split
import numpy as np

@torch.no_grad()
def evaluate_epoch(model, loader, device):
    model.eval()
    tot_loss = 0.0
    tot_correct = 0
    n = 0
    for X, Y, M in loader:
        X, Y, M = X.to(device), Y.to(device), M.to(device)
        logits = model(X, pad_mask=M)
        loss = F.cross_entropy(logits, Y, reduction="sum")
        tot_loss += loss.item()
        tot_correct += (logits.argmax(-1) == Y).sum().item()
        n += X.size(0)
    return (tot_loss / max(n, 1), tot_correct / max(n, 1))

def train_with_split_and_plots(
    samples,
    test_frac=0.2,
    test_samples=None,
    seed=42,
    epochs=30,
    batch_size=64,
    lr=3e-4,
    device=None,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- build datasets ----------
    if test_samples is None:
        # single pool → deterministic split
        full_ds = PrefixNextBitPaddedDataset(samples)

        n_total = len(full_ds)
        n_test = int(round(test_frac * n_total))
        n_train = n_total - n_test

        g = torch.Generator().manual_seed(seed)
        train_ds, test_ds = random_split(full_ds, [n_train, n_test], generator=g)

        print(f"[split] train={n_train} test={n_test}")

    else:
        # explicit train / test sets
        train_ds = PrefixNextBitPaddedDataset(samples)
        test_ds  = PrefixNextBitPaddedDataset(test_samples)

        print(f"[explicit] train={len(train_ds)} test={len(test_ds)}")

    # ---------- loaders ----------
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_pad,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_pad,
    )

    model = PolicyOnlyTransformer(vocab_size=3, pad_id=PAD_ID, max_len=4096).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.01)

    hist = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for ep in range(1, epochs + 1):
        model.train()
        for X, Y, M in train_loader:
            X, Y, M = X.to(device), Y.to(device), M.to(device)
            logits = model(X, pad_mask=M)
            loss = F.cross_entropy(logits, Y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        tr_loss, tr_acc = evaluate_epoch(model, train_loader, device)
        te_loss, te_acc = evaluate_epoch(model, test_loader, device)

        hist["train_loss"].append(tr_loss)
        hist["train_acc"].append(tr_acc)
        hist["test_loss"].append(te_loss)
        hist["test_acc"].append(te_acc)

        print(f"epoch {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | test loss {te_loss:.4f} acc {te_acc:.3f}")

    # plots
    xs = np.arange(1, epochs + 1)

    plt.figure(figsize=(7, 4))
    plt.plot(xs, hist["train_loss"], label="train")
    plt.plot(xs, hist["test_loss"], label="test")
    plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.title("Loss")
    plt.legend(); plt.tight_layout(); plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(xs, hist["train_acc"], label="train")
    plt.plot(xs, hist["test_acc"], label="test")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title("Accuracy")
    plt.legend(); plt.tight_layout(); plt.show()

    return model, (train_ds, test_ds), hist

In [9]:
def create_partial_str(seq):
    s = ""
    for i in range(4):
        cur_seq = seq[i]
        for j in range(len(cur_seq)):
            if cur_seq[j] == 1:
                s += "1"
            else:
                s += "0"
    return s

partial_strs = set()
double_seq = []
for item in original_dataset:
    seq = item[0]
    cur_str = create_partial_str(seq)
    if cur_str in partial_strs:
        double_seq.append(seq[:4])
    else:
        partial_strs.add(cur_str)

print(len(double_seq))

train_dataset, test_dataset = train_test_split(original_dataset, test_size=0.1)

5


In [10]:
count = [0, 0]
for item in train_dataset:
    if item[1] == -1:
        count[0] += 1
    else:
        count[1] += 1
print(count)

count = [0, 0]
for item in test_dataset:
    if item[1] == -1:
        count[0] += 1
    else:
        count[1] += 1
print(count)

model = train_overfit(train_dataset, epochs=20)
eval_dataset = PrefixNextBitPaddedDataset(test_dataset)
evaluate_policy(model, eval_dataset)

NameError: name 'train_dataset' is not defined

In [8]:
from sequence_generator_sat import generate_dataset_sat
from sequence_generator import generate_dataset
import pickle

## Create dataset on other lengths and evaluate on new length

r = 7

original_dataset = generate_dataset_sat(r=7, partial_flat_len=12, limit=8000, max_partial_tries_per_seq=10, verbose=False)

# original_dataset = []
# original_dataset.extend(generate_dataset(r - 2))
# original_dataset.extend(generate_dataset(r - 4))

# with open('original_dataset_9.pkl', 'rb') as file:
#     original_dataset.extend(pickle.load(file))

# with open('original_dataset_11.pkl', 'rb') as file:
#     test_dataset.extend(pickle.load(file))
# test_dataset = test_dataset[:500]

for item in original_dataset:
    if len(item[0][0]) <= 3:
        original_dataset.remove(item)

# for item in test_dataset:
#     if len(item[0][0]) <= 5:
#         test_dataset.remove(item)

count = [0, 0]
for item in original_dataset:
    if item[1] == -1:
        count[0] += 1
    else:
        count[1] += 1
print(count)

# model = train_overfit(original_dataset, epochs=10)
# torch.save(model.state_dict(), f"model_long_11.pt")

model, (train_ds, test_ds), hist = train_with_split_and_plots(original_dataset, epochs=20)
torch.save(model.state_dict(), "model_7_train_test.pt")

[3377, 4221]
[split] train=6078 test=1520
epoch   1 | train loss 0.6495 acc 0.607 | test loss 0.6513 acc 0.609
epoch   2 | train loss 0.5766 acc 0.681 | test loss 0.5660 acc 0.667
epoch   3 | train loss 0.5307 acc 0.711 | test loss 0.5233 acc 0.711
epoch   4 | train loss 0.5109 acc 0.718 | test loss 0.5016 acc 0.713
epoch   5 | train loss 0.4905 acc 0.728 | test loss 0.4892 acc 0.719
epoch   6 | train loss 0.4667 acc 0.745 | test loss 0.4837 acc 0.723
epoch   7 | train loss 0.4674 acc 0.749 | test loss 0.4921 acc 0.732
epoch   8 | train loss 0.4531 acc 0.753 | test loss 0.4691 acc 0.718


KeyboardInterrupt: 

In [8]:
eval_samples = generate_dataset(9)

for item in eval_samples:
    if len(item[0][0]) == 0:
        eval_samples.remove(item)

eval_dataset = PrefixNextBitPaddedDataset(eval_samples)
print("Start evaluation")

evaluate_policy(model, eval_dataset)

TypeError: iteration over a 0-d tensor

In [ ]:
import random

single_example = original_dataset[int(random.random() * len(original_dataset))]
print(single_example)

seq, next_bit = single_example
for s in seq:
    print(s)

# single_eval_dataset = PrefixNextBitPaddedDataset([single_example])
# evaluate_policy(model, single_eval_dataset)

([(1, 1, -1, 1, -1), (1, 1, 1, -1), (1, -1, -1, 1), (1, -1, -1, 1)], 1)
(1, 1, -1, 1, -1)
(1, 1, 1, -1)
(1, -1, -1, 1)
(1, -1, -1, 1)


In [13]:
from cnf_automator import generate_encoding, verify_assignment
import json

def conversion(s):
    s = s.replace('−', '-')
    s = s.split("\n")[1:-1]
    seqs = [seq[3:] for seq in s]
    new_seqs = [json.loads(seq) for seq in seqs]
    return new_seqs

s = """
X: [1, -1, 1, 1, -1, -1, 1, 1, 1, -1, -1]
Y: [-1, 1, 1, -1, 1, -1, -1, 1, -1, 1, 1]
Z: [1, 1, -1, 1, -1, 1, -1, -1, 1, 1, -1]
W: [1, -1, -1, 1, 1, -1, 1, -1, -1, 1, 1]
"""

r = 23
seq = conversion(s)
cnf, varmap, pmap_lags, pool = generate_encoding(r)
sat = verify_assignment(seq, varmap, cnf)
sat

False

In [16]:
import torch

model = PolicyOnlyTransformer()
model.eval()

max_len = 11
half_len = max_len // 2
pad_id = model.pad_id

# Start with an empty sequence (just PADs)
x = torch.full((1, max_len), pad_id, dtype=torch.long)
pad_mask = (x == pad_id)

In [97]:
import random

single_example = ([(1, 1, 1, 1), (1, 1, 1, 1), (1, 1, 1), (1, 1, 1)], 1)

seq, next_bit = single_example
for s in seq:
    print(s)

single_eval_dataset = PrefixNextBitPaddedDataset([single_example])
evaluate_policy(model, single_eval_dataset)

(1, 1, 1, 1)
(1, 1, 1, 1)
(1, 1, 1)
(1, 1, 1)
tensor([1])


{'loss': 0.1296236217021942, 'acc': 1.0, 'n': 1, 'mistakes': None}

In [9]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def predict_next_bit_from_prefix(model, cur_seq, device=None, temperature=1.0, strategy="argmax"):
    """
    cur_seq: list of sequences, e.g. [X, Y, Z, W], each a list/1D tensor of ±1
    returns: dict(pred in {0,1}, conf, probs[2], logits[2])
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    # flatten to 0/1 token stream
    bits = flatten_state(cur_seq)                     # 1D LongTensor of 0/1
    T = bits.numel()
    max_len = model.pos_emb.num_embeddings

    if T > max_len:
        raise ValueError(f"prefix length {T} exceeds model max_len {max_len}")

    # build one-row padded input and mask
    x = torch.full((1, T if T > 0 else 1), PAD_ID, dtype=torch.long)  # allow empty prefix
    if T > 0:
        x[0, :T] = bits
    pad_mask = (x == PAD_ID)

    x, pad_mask = x.to(device), pad_mask.to(device)
    logits = model(x, pad_mask=pad_mask).squeeze(0)   # [2]

    if temperature != 1.0:
        logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)

    if strategy == "sample":
        pred = int(torch.multinomial(probs, 1).item())
    else:
        pred = int(probs.argmax().item())

    return {
        "pred":   pred,
        "conf":   float(probs[pred].item()),
        "probs":  probs.detach().cpu(),   # tensor [2]
        "logits": logits.detach().cpu(),  # tensor [2]
    }

In [ ]:
import random
from cnf_automator import verify_assignment, generate_encoding

r = 7

def add(seq, next_bit):
    first_len = len(seq[0])
    added = False
    for i in range(4):
        if (len(seq[i]) < first_len):
            new_element = (next_bit, )
            new_seq = seq[i] + new_element
            seq[i] = new_seq
            added = True
            break
    if not added:
        new_element = (next_bit, )
        new_seq = seq[0] + new_element
        seq[0] = new_seq
    return seq


seq = [(1, ), (1, ), (1, ), (1, )]
cnf, varmap, pmap_lags, pool = generate_encoding(r)
sat, core = verify_assignment(seq, varmap, cnf)

rev_varmap = {}
for k, v in varmap.items():
    rev_varmap[v] = k

model = PolicyOnlyTransformer()
model.load_state_dict(torch.load(f"model_7_train_test.pt", weights_only=True))
model.eval()

odds_one = 0.52
num_unsat, sum_unsat = 0, 0
count_one, count_total = 0, 0
count_right, count_wrong = 0, 0
pred_one = 0
first_incorrect = -1
for i in range(4 * (r - 1)):
    res = predict_next_bit_from_prefix(model, seq)
    next_bit = res["pred"]
    # next_bit = random.choices([0, 1], weights=[1 - odds_one, odds_one], k=1)[0]
    prev_seq = seq.copy()
    seq = add(seq, next_bit)
    sat, core = verify_assignment(seq, varmap, cnf)
    # if next_bit == 1:
    #     pred_one += 1
    # count_total += 1
    if core:
    #     if first_incorrect == -1:
    #         first_incorrect = i + 4
    #     if next_bit == 0:
    #         count_one += 1
    #     num_unsat += 1
    #     sum_unsat += len(core)
    #     decoded_core = []
    #     for num in core:
    #         neg = "-" if num < 0 else ""
    #         decoded_core.append(neg + rev_varmap[abs(num)])
    #     print(i, decoded_core)
        seq = prev_seq.copy()
        seq = add(seq, 1 - next_bit)
    # else:
        # if next_bit == 1:
        #     count_one += 1
        # print("RIGHT:", next_bit, max(res["probs"]))
    sec_seq = add(prev_seq, 1 - next_bit)
    sat, core_sec = verify_assignment(sec_seq, varmap, cnf)
    if core or core_sec:
        print(i, "ONLY ONE CHOICE WORKS")
        if core:
            print("WRONG")
            count_wrong += 1
        else:
            print("RIGHT")
            count_right += 1

print(count_right, count_wrong)
print(pred_one, count_one, count_total)
print(seq)
print(first_incorrect)
print(num_unsat)
print(sum_unsat)

12 ONLY ONE CHOICE WORKS
RIGHT
13 ONLY ONE CHOICE WORKS
RIGHT
14 ONLY ONE CHOICE WORKS
RIGHT
15 ONLY ONE CHOICE WORKS
RIGHT
17 ONLY ONE CHOICE WORKS
RIGHT
18 ONLY ONE CHOICE WORKS
RIGHT
19 ONLY ONE CHOICE WORKS
RIGHT
20 ONLY ONE CHOICE WORKS
WRONG
21 ONLY ONE CHOICE WORKS
RIGHT
22 ONLY ONE CHOICE WORKS
RIGHT
23 ONLY ONE CHOICE WORKS
RIGHT
10 1
0 0 0
[(1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 1, 1, 1, 0), (1, 1, 0, 0, 1, 0, 1), (1, 0, 0, 0, 0, 1, 0)]
-1
0
0


In [27]:
from cnf_automator import verify_assignment, generate_encoding
import random

models = ["model_9_9.pt", "model_11_11.pt", "model_11_11_9.pt", "model_11_9_7.pt", "model_11_11_9_20.pt", "model_9.pt", "model_long_9.pt"]

def add(seq, next_bit):
    first_len = len(seq[0])
    added = False
    for i in range(4):
        if (len(seq[i]) < first_len):
            new_element = (next_bit, )
            new_seq = seq[i] + new_element
            seq[i] = new_seq
            added = True
            break
    if not added:
        new_element = (next_bit, )
        new_seq = seq[0] + new_element
        seq[0] = new_seq
    return seq

def evaluate_model(model_name, r):
    seq = [(1, ), (1, ), (1, ), (1, )]
    cnf, varmap, pmap_lags, pool = generate_encoding(r)
    sat, core = verify_assignment(seq, varmap, cnf)

    rev_varmap = {}
    for k, v in varmap.items():
        rev_varmap[v] = k

    model = PolicyOnlyTransformer()
    model.load_state_dict(torch.load(model_name, weights_only=True))
    model.eval()

    num_unsat, sum_unsat = 0, 0
    correct_one, incorrect_one = 0, 0
    correct_neg, incorrect_neg = 0, 0
    first_incorrect = -1
    for i in range(4 * (r - 1)):
        res = predict_next_bit_from_prefix(model, seq)
        next_bit = res["pred"]
        prev_seq = seq.copy()
        seq = add(seq, next_bit)
        sat, core = verify_assignment(seq, varmap, cnf)
        if core:
            if first_incorrect == -1:
                first_incorrect = i + 4
            if next_bit == 0:
                incorrect_neg += 1
            if next_bit == 1:
                incorrect_one += 1
            num_unsat += 1
            sum_unsat += len(core)
            decoded_core = []
            for num in core:
                neg = "-" if num < 0 else ""
                decoded_core.append(neg + rev_varmap[abs(num)])
            seq = prev_seq
            seq = add(seq, 1 - next_bit)
        else:
            if next_bit == 1:
                correct_one += 1
            else:
                correct_neg += 1
    return [
        correct_one, 
        correct_neg, 
        incorrect_one,
        incorrect_neg,
        seq,
        first_incorrect,
        num_unsat,
        sum_unsat
    ]

def evaluate_models(r):
    for model in models:
        print("Evaluating on", model)
        correct_one, correct_neg, incorrect_one, incorrect_neg, seq, first_incorrect, num_unsat, sum_unsat = evaluate_model(model, r)
        print(correct_one, correct_neg, incorrect_one, incorrect_neg)
        print(seq)
        print(first_incorrect)
        print(num_unsat, sum_unsat)
        print()

r = 17
evaluate_models(r)

Evaluating on model_9_9.pt
36 11 16 1
[(1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0), (1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0), (1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1)]
32
17 802

Evaluating on model_11_11.pt
28 14 17 5
[(1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0), (1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0)]
30
22 897

Evaluating on model_11_11_9.pt
32 16 14 2
[(1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0), (1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0), (1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1)]
29
16 718

Evaluating on model_11_9_7.pt
35 10 17 2
[(1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0), (1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0), (1, 1, 